# Семинар 10. Тестирование и качество проекта

Семинар справочный: обязательной сдаваемой части нет. Разбираем рабочий минимум `pytest`, параметризацию, fixtures, mocking, логирование и конфигурацию, затем применяем его к проблемам групповых проектов.

## Цели

- отделять предметную логику от ввода-вывода;
- писать один ясный тест на один контракт;
- объединять однотипные случаи параметризацией;
- использовать fixture для подготовки и очистки окружения;
- подменять внешнюю зависимость fake или mock;
- логировать события без `print` в библиотечном коде;
- читать конфигурацию на границе приложения;
- локализовать проблему проекта минимальным воспроизводимым примером.

## Перед началом

Для запуска примеров в собственном проекте установите dev-зависимости из занятия 4, включая `pytest`. Ячейки `exercise` здесь необязательны: их можно адаптировать к предметному ядру проекта. Команды запускаются из корня установленного проекта, а не через изменение `PYTHONPATH` вручную.

## Минимальная форма теста

Тест строится как Arrange — Act — Assert:

1. подготовить вход и зависимости;
2. выполнить одно наблюдаемое действие;
3. проверить результат или ошибку.

Название описывает поведение, например `test_withdraw_rejects_amount_above_balance`, а не номер задания. Тест не должен повторять внутренний алгоритм функции.

In [ ]:
from decimal import Decimal

def net_amount(amount: Decimal, fee: Decimal) -> Decimal:
    if amount <= 0 or fee < 0 or fee > amount:
        raise ValueError("invalid amount or fee")
    return amount - fee

# TODO: запишите три обычных assert для типичного и граничных случаев.

## Тот же контракт в `pytest`

Файл `tests/test_money.py`:

```python
from decimal import Decimal

from project_name.money import net_amount


def test_net_amount_subtracts_fee() -> None:
    result = net_amount(Decimal("100"), Decimal("7"))
    assert result == Decimal("93")
```

Запуск `pytest -q` находит функции `test_*`. Падающий `assert` показывает различие значений и traceback.

## Проверка ошибки через `pytest.raises`

Исключение является частью контракта, его не надо ловить вручную внутри теста:

```python
import pytest


def test_net_amount_rejects_fee_above_amount() -> None:
    with pytest.raises(ValueError, match="invalid amount"):
        net_amount(Decimal("10"), Decimal("11"))
```

Проверяйте тип и устойчивую смысловую часть сообщения, а не весь текст с несущественными деталями.

## Параметризация вместо копирования тестов

Если шаги одинаковы, а меняются только вход и ответ:

```python
import pytest

@pytest.mark.parametrize(
    ("amount", "fee", "expected"),
    [
        (Decimal("100"), Decimal("0"), Decimal("100")),
        (Decimal("100"), Decimal("7"), Decimal("93")),
        (Decimal("0.01"), Decimal("0.01"), Decimal("0")),
    ],
)
def test_net_amount_cases(amount, fee, expected) -> None:
    assert net_amount(amount, fee) == expected
```

Параметризация показывает таблицу контракта. Не объединяйте ею сценарии с разными причинами падения и разной подготовкой.

In [ ]:
cases = [
    (Decimal("100"), Decimal("0"), Decimal("100")),
    (Decimal("100"), Decimal("7"), Decimal("93")),
]

# TODO: сначала выполните таблицу обычным циклом с assert,
# затем перенесите её в @pytest.mark.parametrize в файле проекта.

## Fixture: подготовка с жизненным циклом

Fixture полезна для переиспользуемого окружения, особенно если после теста нужна очистка:

```python
import pytest

@pytest.fixture
def transactions_file(tmp_path):
    path = tmp_path / "transactions.csv"
    path.write_text("t-1;food;100\n", encoding="utf-8")
    return path

def test_csv_source_reads_transaction(transactions_file):
    source = CsvTransactionSource(transactions_file.read_text().splitlines())
    assert list(source.load())[0].transaction_id == "t-1"
```

Встроенная `tmp_path` даёт отдельный временный каталог каждому тесту. Fixture не нужна для одной константы, которую проще создать прямо в тесте.

In [ ]:
def parse_ids(lines):
    return [line.split(";", 1)[0] for line in lines if line.strip()]

# TODO в проекте: fixture на временный CSV с двумя строками.
# TODO: тест обычного чтения и тест пустого файла.

## Fake или mock

**Fake** — маленькая рабочая реализация интерфейса, например источник транзакций из списка. Она хороша, когда тесту важно состояние и итоговое значение.

**Mock** программируется ожидаемыми ответами и записывает вызовы. Он нужен, когда контракт включает взаимодействие: метод вызван один раз с конкретным аргументом.

Не подменяйте `Decimal`, списки и простые чистые функции. Обычно подменяют медленную или нестабильную границу: сеть, время, файловую систему, внешний сервис.

In [ ]:
class MemorySource:
    def __init__(self, rows):
        self._rows = tuple(rows)

    def load(self):
        return iter(self._rows)

# TODO: передайте MemorySource в сервис проекта и проверьте отчёт без файлов.

## `unittest.mock` для проверки взаимодействия

Стандартная библиотека уже содержит `Mock`:

```python
from unittest.mock import Mock

source = Mock()
source.load.return_value = [transaction]
service = ReportService(source)

report = service.build()

assert report.total() == Decimal("100")
source.load.assert_called_once_with()
```

Если тест проверяет десять внутренних вызовов, он привязывается к реализации и мешает рефакторингу. Проверяйте только значимое взаимодействие на границе.

In [ ]:
from unittest.mock import Mock

notifier = Mock()
notifier.send.return_value = "message-1"
message_id = notifier.send("report ready")
assert message_id == "message-1"
notifier.send.assert_called_once_with("report ready")

# TODO: замените notifier на реальную внешнюю зависимость вашего сервиса.

## Патчить там, где имя используется

Если `services.py` содержит `from gateways import send`, код обращается к имени `services.send`. Поэтому patch направляют на `project_name.services.send`, а не на место первоначального определения.

Ещё лучше передать gateway в конструктор через протокол. Тогда большинство тестов обходится без строкового пути patch и остаётся устойчивым к перемещению модулей.

## Логирование вместо диагностического `print`

Модуль получает именованный logger, но не настраивает глобальный вывод:

```python
import logging

logger = logging.getLogger(__name__)

def build_report(source):
    logger.info("building report")
    try:
        return ReportService(source).build()
    except ValueError:
        logger.exception("invalid transaction data")
        raise
```

Точка входа приложения выбирает уровень и формат. `DEBUG` — диагностические детали, `INFO` — нормальные значимые события, `WARNING` — отклонение с продолжением, `ERROR` — неуспешная операция.

In [ ]:
import logging

logger = logging.getLogger("lesson10.demo")

def divide_total(total, count):
    if count == 0:
        logger.warning("cannot calculate average: empty report")
        return None
    return total / count

# TODO в pytest: используйте caplog и проверьте уровень и смысл сообщения.

## Проверка логов через `caplog`

```python
import logging

def test_empty_report_is_logged(caplog):
    with caplog.at_level(logging.WARNING):
        result = divide_total(0, 0)

    assert result is None
    assert "empty report" in caplog.text
```

Не проверяйте timestamp и полный формат строки: они относятся к настройке приложения, а не к смыслу события.

## Конфигурация на границе

Переменная окружения является строкой или отсутствует. Преобразуйте и проверьте её один раз при запуске, затем передавайте готовое значение:

```python
import os

raw_timeout = os.environ.get("APP_TIMEOUT", "10")
timeout = int(raw_timeout)
if timeout <= 0:
    raise ValueError("APP_TIMEOUT must be positive")
```

Предметный сервис не должен заново читать окружение в каждом методе. Явный параметр легче тестировать и объяснять.

In [ ]:
def parse_timeout(raw_value):
    value = 10 if raw_value is None else int(raw_value)
    if value <= 0:
        raise ValueError("timeout must be positive")
    return value

assert parse_timeout(None) == 10
assert parse_timeout("25") == 25
# TODO: проверки нуля, отрицательного и нечислового значения.

## Разбор проблемы проекта

Приносите не весь репозиторий со словами «не работает», а минимальный маршрут:

1. ожидаемый результат;
2. фактический результат и полный traceback;
3. минимальный вход;
4. функция или два взаимодействующих объекта;
5. команда точного воспроизведения;
6. уже проверенные гипотезы.

Сначала определите слой сбоя: предметная логика, разбор данных, внешняя зависимость, конфигурация или упаковка. Затем замените внешние границы маленькими fake и сузьте проблему.

## Контрольная точка проекта после занятия 10

- Предметное ядро не зависит от CLI, бота или HTTP.
- Основные значения и инварианты представлены понятными функциями или классами.
- Внешние зависимости передаются явно.
- Есть автоматические тесты обычных сценариев и ошибок.
- `pytest` запускается одной командой из корня.
- Аннотации не противоречат фактическим значениям.
- Диагностические события идут в logger, а не в случайные `print`.
- Конфигурация проверяется при запуске.
- README содержит воспроизводимые команды установки и проверки.

## Мини-консультация

Выберите один текущий риск проекта и оформите его как проверяемый вопрос:

- Какой объект или функция отвечает за правило?
- Какая зависимость мешает тесту?
- Можно ли заменить её fake через узкий интерфейс?
- Какой минимальный тест докажет исправление?
- Какое лог-сообщение поможет отличить эту ошибку в запуске?

Ячейка ниже — необязательный шаблон фиксации решения.

In [ ]:
project_issue = {
    "expected": None,
    "actual": None,
    "minimal_input": None,
    "suspected_layer": None,
    "dependency_to_replace": None,
    "test_to_add": None,
}

# TODO по желанию: заполните словарь для одной реальной проблемы проекта.

## Самопроверка

- Можете сформулировать тест как Arrange — Act — Assert?
- Можете отличить таблицу случаев для parametrize от разных сценариев?
- Можете объяснить, когда fixture полезнее локальной переменной?
- Можете выбрать fake или mock по наблюдаемому контракту?
- Можете объяснить, где patch ищет имя?
- Можете выбрать уровень логирования для события?
- Можете вынести чтение окружения на границу приложения?
- Можете сократить проблему проекта до одного воспроизводимого теста?

## Источники

- [pytest: parametrizing tests](https://docs.pytest.org/en/stable/how-to/parametrize.html) — таблицы тестовых случаев.
- [pytest: fixtures](https://docs.pytest.org/en/stable/how-to/fixtures.html) — подготовка и жизненный цикл окружения.
- [`unittest.mock`](https://docs.python.org/3/library/unittest.mock.html) — mocks, assertions и patch.
- [Logging HOWTO](https://docs.python.org/3/howto/logging.html) — loggers, уровни и настройка.
- [`os.environ`](https://docs.python.org/3/library/os.html#os.environ) — переменные окружения как отображение строк.

## Итоги

Справочный семинар связал объектную модель с эксплуатацией проекта: инварианты проверяются тестами, повторяющиеся случаи задаются таблицей, fixtures управляют окружением, fake и mock изолируют внешние границы. Логи описывают события, конфигурация проверяется при запуске, а проблема проекта превращается в минимальный воспроизводимый тест вместо ручной серии действий.